<a href="https://colab.research.google.com/github/umanahekeobong-cell/Crack-Detector/blob/main/23_EG_EE_057_Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset preparation and preprocessing

In [12]:
# Verify Required Libraries
!pip show matplotlib seaborn scikit-learn tensorflow

Name: matplotlib
Version: 3.10.0
Summary: Python plotting package
Home-page: https://matplotlib.org
Author: John D. Hunter, Michael Droettboom
Author-email: Unknown <matplotlib-users@python.org>
License: License agreement for matplotlib versions 1.3.0 and later

 1. This LICENSE AGREEMENT is between the Matplotlib Development Team
 ("MDT"), and the Individual or Organization ("Licensee") accessing and
 otherwise using matplotlib software in source or binary form and its
 associated documentation.

 2. Subject to the terms and conditions of this License Agreement, MDT
 hereby grants Licensee a nonexclusive, royalty-free, world-wide license
 to reproduce, analyze, test, perform and/or display publicly, prepare
 derivative works, distribute, and otherwise use matplotlib
 alone or in any derivative version, provided, however, that MDT's
 License Agreement and MDT's notice of copyright, i.e., "Copyright (c)
 2012- Matplotlib Development Team; All Rights Reserved" are retained in
 matplotlib

In [13]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import (
accuracy_score, precision_score, recall_score,
f1_score, classification_report, confusion_matrix
)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
print(f'TensorFlow version : {tf.__version__}')
print(f'Random seed set to : {SEED}')

TensorFlow version : 2.20.0
Random seed set to : 42


In [14]:
results_dir = "results/lab7/"
if not os.path.exists(results_dir):
  os.makedirs(results_dir, exist_ok=True)

In [15]:
!nvidia-smi
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  print(f'\nGPUs available: {len(gpus)}')
  for gpu in gpus:
      print(f' - {gpu}')
else:
  print('\nNo GPU found — training will run on CPU (slower).')

Wed Jul 29 19:40:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
import os
import shutil
from pathlib import Path

# Updated base_input to point to the directory where the zip file was unzipped
base_input = "/content/drive/MyDrive/extracted_dataset/"

# Define the base output directory for renamed files, also in Google Drive
# You can change 'PlantVillage_Dataset_Renamed' to another name if preferred
base_output = "/content/drive/MyDrive/concrete crack renamed/"

# Ensure the base_output directory exists
os.makedirs(base_output, exist_ok=True)

base_path = Path(base_output)
print(f"Base input path: {base_input}")
print(f"Base output path: {base_output}")
base_path

Base input path: /content/drive/MyDrive/extracted_dataset/
Base output path: /content/drive/MyDrive/concrete crack renamed/


PosixPath('/content/drive/MyDrive/concrete crack renamed')

In [11]:
import random

# Define the desired splits for the output dataset and their ratios
splits_config = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
} # Ensure these sum up to 1 (or close to 1 due to integer division)

print(f"Processing dataset from: {base_input}")
print(f"Outputting to: {base_output}")

# Ensure the base output split directories exist
for split_name in splits_config.keys():
    os.makedirs(os.path.join(base_output, split_name), exist_ok=True)

# Iterate through top-level categories (e.g., Decks, Pavements, Walls)
for category_name in os.listdir(base_input):
    category_path = os.path.join(base_input, category_name)
    if not os.path.isdir(category_path):
        print(f"Skipping non-directory item in base_input: {category_path}")
        continue

    # Iterate through class folders within each category (e.g., healthy, rust)
    for class_folder_name in os.listdir(category_path):
        class_folder_path = os.path.join(category_path, class_folder_name)
        if not os.path.isdir(class_folder_path):
            print(f"Skipping non-directory item in category_path: {class_folder_path}")
            continue

        # Create the combined class label for the output
        final_class_label = f"{category_name}_{class_folder_name}"

        # Get all image files in the current class_folder_path
        all_files = [f for f in os.listdir(class_folder_path) if os.path.isfile(os.path.join(class_folder_path, f))]
        if not all_files:
            print(f"  No files found in {class_folder_path}. Skipping this class.")
            continue

        print(f"  Processing class: {final_class_label} with {len(all_files)} files")

        # Shuffle files for random splitting
        random.shuffle(all_files)

        # Calculate split sizes
        num_total = len(all_files)
        num_train = int(num_total * splits_config["train"])
        num_val = int(num_total * splits_config["val"])
        num_test = num_total - num_train - num_val # Remaining files go to test to ensure all are used

        # Assign files to splits
        train_files = all_files[:num_train]
        val_files = all_files[num_train : num_train + num_val]
        test_files = all_files[num_train + num_val :]

        file_assignments = {
            "train": train_files,
            "val": val_files,
            "test": test_files
        }

        # Copy files to their respective destination directories
        for split_name, files_to_copy in file_assignments.items():
            dst_dir = os.path.join(base_output, split_name, final_class_label)
            os.makedirs(dst_dir, exist_ok=True)

            for file_name in files_to_copy:
                src_file = os.path.join(class_folder_path, file_name)
                shutil.copy(src_file, os.path.join(dst_dir, file_name))

print("Data reorganization complete!")

Processing dataset from: /content/drive/MyDrive/extracted_dataset/
Outputting to: /content/drive/MyDrive/concrete crack renamed/
  Processing class: Decks_Cracked with 2025 files
  Processing class: Decks_Non-cracked with 11595 files
  Processing class: Pavements_Cracked with 2608 files
  Processing class: Pavements_Non-cracked with 6111 files
Data reorganization complete!


In [17]:
BASE_DIR = Path(base_output)
train_dir = BASE_DIR / 'train'
val_dir = BASE_DIR / 'val'
test_dir = BASE_DIR / 'test'


In [19]:
for split, d in [('train', train_dir), ('val', val_dir), ('test',
    test_dir)]:
    exists = d.exists()
    count = sum(1 for _ in d.rglob('*.jpg')) + sum(1 for _ in
    d.rglob('*.png')) if exists else 0
    print(f'{split:>5} dir exists={exists} images≈{count}')

train dir exists=True images≈20507
  val dir exists=True images≈6087
 test dir exists=True images≈5850
